# 03 — Evaluation and error analysis

This notebook evaluates validated predictions against the toy gold standard.

It deliberately keeps format/alignment validity separate from linguistic quality.

In [ ]:
!pip -q install scikit-learn matplotlib

In [ ]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt

PROJECT_DIR = Path('/content/lrec2026_llm_annotator')
DATA_DIR = PROJECT_DIR / 'data' / 'sample'
OUTPUT_DIR = PROJECT_DIR / 'outputs'

validated_path = OUTPUT_DIR / 'validated_predictions.jsonl'
if not validated_path.exists():
    raise FileNotFoundError('Run 02_structured_outputs_and_validation.ipynb first.')

validated = pd.read_json(validated_path, lines=True)
gold = pd.read_csv(DATA_DIR / 'toy_sentences.csv')
for col in ['tokens', 'gold_pos', 'gold_lemma', 'gold_features']:
    gold[col] = gold[col].apply(json.loads)

gold_by_id = gold.set_index('id').to_dict(orient='index')
validated[['sentence_id','mode','is_valid']].head()

## Flatten token-level predictions

Only valid, aligned outputs are used for linguistic scores. Invalid outputs remain counted separately.

In [ ]:
def flatten_predictions(validated_df):
    rows = []
    for _, rec in validated_df.iterrows():
        sid = rec['sentence_id']
        g = gold_by_id[sid]
        if not rec['is_valid']:
            continue
        parsed = rec['parsed']
        for i, pred_tok in enumerate(parsed['tokens']):
            rows.append({
                'sentence_id': sid,
                'language': rec['language'],
                'mode': rec['mode'],
                'token_index': i,
                'surface': g['tokens'][i],
                'gold_pos': g['gold_pos'][i],
                'pred_pos': pred_tok.get('upos'),
                'gold_lemma': g['gold_lemma'][i],
                'pred_lemma': pred_tok.get('lemma'),
                'gold_features': g['gold_features'][i],
                'pred_features': pred_tok.get('features', {}),
                'confidence': pred_tok.get('confidence'),
                'comment': pred_tok.get('comment')
            })
    return pd.DataFrame(rows)

tok = flatten_predictions(validated)
tok.head(10)

In [ ]:
validity_summary = validated.groupby(['mode','is_valid']).size().reset_index(name='n')
validity_summary

## POS and lemma scores

In [ ]:
def safe_mean(x):
    return float(np.mean(x)) if len(x) else np.nan

summary = []
for mode, sub in tok.groupby('mode'):
    summary.append({
        'mode': mode,
        'n_tokens_evaluated': len(sub),
        'pos_accuracy': safe_mean(sub['gold_pos'] == sub['pred_pos']),
        'lemma_exact_match': safe_mean(sub['gold_lemma'] == sub['pred_lemma']),
    })
summary = pd.DataFrame(summary)
summary

In [ ]:
by_lang = tok.assign(
    pos_correct=lambda x: x['gold_pos'] == x['pred_pos'],
    lemma_correct=lambda x: x['gold_lemma'] == x['pred_lemma']
).groupby(['mode','language']).agg(
    n_tokens=('surface','size'),
    pos_accuracy=('pos_correct','mean'),
    lemma_exact_match=('lemma_correct','mean')
).reset_index()
by_lang

## Morphological feature F1

We score feature-value pairs. For each token, compare sets like `Case=Nom` and `Number=Sing`.

In [ ]:
def feature_set(d):
    if not isinstance(d, dict):
        return set()
    return {f'{k}={v}' for k, v in d.items() if v not in [None, '', 'None']}

def prf(tp, fp, fn):
    p = tp / (tp + fp) if tp + fp else 0.0
    r = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2*p*r/(p+r) if p+r else 0.0
    return p, r, f1

rows = []
for mode, sub in tok.groupby('mode'):
    tp = fp = fn = 0
    for _, row in sub.iterrows():
        g = feature_set(row['gold_features'])
        p = feature_set(row['pred_features'])
        tp += len(g & p)
        fp += len(p - g)
        fn += len(g - p)
    prec, rec, f1 = prf(tp, fp, fn)
    rows.append({'mode': mode, 'feature_precision': prec, 'feature_recall': rec, 'feature_f1': f1, 'tp': tp, 'fp': fp, 'fn': fn})
feat_summary = pd.DataFrame(rows)
feat_summary

## Confusion matrix

In [ ]:
mode_to_plot = 'zero_shot'
sub = tok[tok['mode'] == mode_to_plot]
labels = sorted(set(sub['gold_pos']) | set(sub['pred_pos']))
cm = confusion_matrix(sub['gold_pos'], sub['pred_pos'], labels=labels)

fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(cm)
ax.set_xticks(range(len(labels)))
ax.set_yticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=45, ha='right')
ax.set_yticklabels(labels)
ax.set_xlabel('Predicted')
ax.set_ylabel('Gold')
ax.set_title(f'POS confusion matrix — {mode_to_plot}')
for i in range(len(labels)):
    for j in range(len(labels)):
        ax.text(j, i, cm[i, j], ha='center', va='center')
fig.tight_layout()
fig_path = OUTPUT_DIR / f'pos_confusion_{mode_to_plot}.png'
fig.savefig(fig_path, dpi=160)
print('Saved', fig_path)
plt.show()

## Token-level error table

In [ ]:
def classify_error(row):
    if row['gold_pos'] != row['pred_pos']:
        return 'pos'
    if row['gold_lemma'] != row['pred_lemma']:
        return 'lemma'
    if feature_set(row['gold_features']) != feature_set(row['pred_features']):
        return 'morphology'
    return 'correct'

errors = tok.copy()
errors['error_type'] = errors.apply(classify_error, axis=1)
errors = errors[errors['error_type'] != 'correct'].copy()
errors[['sentence_id','language','mode','surface','gold_pos','pred_pos','gold_lemma','pred_lemma','error_type','confidence','comment']]

In [ ]:
summary_path = OUTPUT_DIR / 'summary_metrics.csv'
errors_path = OUTPUT_DIR / 'token_errors.csv'

summary.merge(feat_summary, on='mode').to_csv(summary_path, index=False, encoding='utf-8')
errors.to_csv(errors_path, index=False, encoding='utf-8')
print('Wrote', summary_path)
print('Wrote', errors_path)

## Participant TODO

Choose one error. Decide whether it is:

- a true linguistic error;
- a convention mismatch;
- caused by the prompt;
- caused by an unclear tagset;
- caused by an input/tokenisation issue.

Continue with `04_sampling_and_bootstrapping.ipynb`.